# 🧪 Resume Genie Agent — Testing Notebook

## Purpose
Interactively test every routing path, tool chain, and edge case of the LangGraph agent.

## Test Categories
1. **Single Tool** — Resume Checker, Resume Scorer, Cover Letter, Career Coach
2. **Multi-Tool Chaining** — Full analysis (checker + scorer + cover letter)
3. **Edge Cases** — Missing JD, conversation memory, ambiguous queries

---
## 🔧 Part 1: Environment Setup

In [ ]:
# ============================================================================
# OPTIONAL: Install Required Packages
# ============================================================================
# !pip install -qq langchain_groq langchain_community langchain_core langgraph pypdf python-dotenv

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================
import os
import sys
import time
from dotenv import load_dotenv

# Add project root to path so agent package is importable
sys.path.insert(0, os.path.abspath(".."))

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in .env"

print("✅ Environment loaded")

In [ ]:
# ============================================================================
# INITIALIZE LLM + GRAPH
# ============================================================================
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader
from agent.graph import build_graph, TOOL_LABELS

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
graph = build_graph(llm)

print(f"✅ Graph compiled with nodes: {list(graph.get_graph().nodes.keys())}")

In [ ]:
# ============================================================================
# LOAD SAMPLE RESUME
# ============================================================================
loader = PyPDFLoader("../content/Resume.pdf")
documents = loader.load()
resume_text = "\n\n".join(doc.page_content for doc in documents)

print(f"📄 Loaded {len(documents)} page(s), {len(resume_text)} chars")

In [ ]:
# ============================================================================
# SAMPLE JOB DESCRIPTION
# ============================================================================
job_description = """
What You'll Bring:

Bachelor's or higher in a quantitative field (e.g., statistics, mathematics,
computer science, engineering) with a strong academic record.
8 years or above experience in supporting clients and at least 4 years in
analytics, ideally in industries like financial services, insurance, or
digital marketing.
Proven success in end-to-end model development for at least 4 years.
Strong analytical, critical thinking, and problem-solving skills with a
proactive mindset.
Advanced programming skills and aptitude: proficiency with statistical
programs such as R or Python; experience with other programming languages
(SQL, C/C++, Java, Ab Initio) and platforms (Hive, Spark)
Capable of leading complex analytics projects with minimal supervision and
cross-functional collaboration.
Excellent project/time management; able to handle multiple initiatives in a
fast-paced environment.
Strong business acumen and communication skills; able to influence
stakeholders at all levels.
Deep understanding of industry trends and customer needs to identify business
opportunities.
Skilled in translating technical insights into actionable business
recommendations.
Occasional travel required.
"""

print("📋 Job description loaded")

In [ ]:
# ============================================================================
# HELPER: Run a query through the graph and print results
# ============================================================================
def run_query(query, resume=resume_text, jd=None, history=None):
    """Run a query through the agent graph and print node-by-node execution."""
    messages = list(history) if history else []
    messages.append(HumanMessage(content=query))

    state = {
        "messages": messages,
        "resume_text": resume,
        "job_description": jd,
        "tool_choice": None,
        "tool_chain": None,
        "tool_outputs": None,
    }

    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"JD: {'provided' if jd else 'NOT provided'}")
    print(f"{'='*70}")

    start = time.time()
    ai_message = None
    nodes_visited = []

    for event in graph.stream(state, stream_mode="updates"):
        for node_name, node_output in event.items():
            elapsed = time.time() - start
            nodes_visited.append(node_name)

            # Print routing decision from router
            if node_name == "router":
                choice = node_output.get("tool_choice", "?")
                chain = node_output.get("tool_chain")
                chain_str = f" chain={chain}" if chain else ""
                print(f"  [{elapsed:5.1f}s] ROUTER → {choice}{chain_str}")
            elif node_name == "responder":
                print(f"  [{elapsed:5.1f}s] RESPONDER")
            else:
                label = TOOL_LABELS.get(node_name, node_name)
                print(f"  [{elapsed:5.1f}s] TOOL: {label}")

            # Capture AI message
            if "messages" in node_output:
                for msg in node_output["messages"]:
                    if isinstance(msg, AIMessage):
                        ai_message = msg

    total = time.time() - start
    print(f"\nRoute: {' → '.join(nodes_visited)}")
    print(f"Total time: {total:.1f}s")

    if ai_message:
        print(f"Response length: {len(ai_message.content)} chars")
        print(f"\n--- Response Preview (first 500 chars) ---")
        print(ai_message.content[:500])
        if len(ai_message.content) > 500:
            print("...")
    else:
        print("\n⚠️  No AI message returned")

    return ai_message, messages

print("✅ Helper function ready")

---
## 🔍 Part 2: Single Tool Tests — Resume Checker (no JD)

In [ ]:
# ============================================================================
# TEST 1.1: Resume Checker — "Evaluate my resume"
# Expected: router → resume_checker → responder
# ============================================================================
msg, _ = run_query("Evaluate my resume")

In [ ]:
# ============================================================================
# TEST 1.2: Resume Checker — "Review my resume for quality"
# Expected: router → resume_checker → responder
# ============================================================================
msg, _ = run_query("Review my resume for quality")

---
## 📊 Part 3: Single Tool Tests — Resume Scorer (with JD)

In [ ]:
# ============================================================================
# TEST 2.1: Resume Scorer — "Score my resume against this JD"
# Expected: router → resume_scorer → responder
# ============================================================================
msg, _ = run_query("Score my resume against this job description", jd=job_description)

In [ ]:
# ============================================================================
# TEST 2.2: Resume Scorer — "How well does my resume match?"
# Expected: router → resume_scorer → responder
# ============================================================================
msg, _ = run_query("How well does my resume match this JD?", jd=job_description)

---
## ✉️ Part 4: Single Tool Tests — Cover Letter (with JD)

In [ ]:
# ============================================================================
# TEST 3.1: Cover Letter — "Write me a cover letter"
# Expected: router → cover_letter → responder
# ============================================================================
msg, _ = run_query("Write me a cover letter for this role", jd=job_description)

---
## 💬 Part 5: Single Tool Tests — Career Coach (no JD)

In [ ]:
# ============================================================================
# TEST 4.1: Career Coach — Interview prep
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("How should I prepare for a data engineering interview?")

In [ ]:
# ============================================================================
# TEST 4.2: Career Coach — Salary guidance
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("What salary should I expect for my next role?")

---
## 🔗 Part 6: Multi-Tool Chaining (with JD)

In [ ]:
# ============================================================================
# TEST 5.1: Full Analysis — chains all 3 tools
# Expected: router → resume_checker → resume_scorer → cover_letter → responder
# ============================================================================
msg, _ = run_query("Give me a full analysis for this job", jd=job_description)

In [ ]:
# ============================================================================
# TEST 5.2: Full Analysis — alternate phrasing
# Expected: router → resume_checker → resume_scorer → cover_letter → responder
# ============================================================================
msg, _ = run_query("Prepare me completely for this job application", jd=job_description)

---
## 🔗 Part 7: Multi-Tool Chaining WITHOUT JD

In [ ]:
# ============================================================================
# TEST 6.1: Full analysis without JD
# Expected: router → resume_checker → responder (JD tools filtered out)
# ============================================================================
msg, _ = run_query("Give me a full analysis", jd=None)

---
## ⚠️ Part 8: Edge Cases — Missing JD

In [ ]:
# ============================================================================
# TEST 7.1: Cover letter without JD
# Expected: router → END with "need JD" message
# ============================================================================
msg, _ = run_query("Write a cover letter", jd=None)

In [ ]:
# ============================================================================
# TEST 7.2: Score without JD
# Expected: router → END with "need JD" message
# ============================================================================
msg, _ = run_query("Score my resume against the JD", jd=None)

---
## 🧠 Part 9: Conversation Memory

In [ ]:
# ============================================================================
# TEST 8: Multi-turn conversation memory
# The follow-up should reference the first answer
# ============================================================================
print("--- Turn 1 ---")
msg1, history = run_query("What are my main weaknesses?")

# Add the AI response to history for the follow-up
if msg1:
    history.append(msg1)

print("\n\n--- Turn 2 (follow-up) ---")
msg2, _ = run_query("How do I fix those?", history=history)

---
## 🤷 Part 10: Ambiguous / General Queries

In [ ]:
# ============================================================================
# TEST 9.1: General greeting
# Expected: router → career_coach → END (should not crash)
# ============================================================================
msg, _ = run_query("Hello")

In [ ]:
# ============================================================================
# TEST 9.2: Capability question
# Expected: router → career_coach → END
# ============================================================================
msg, _ = run_query("What can you do?")

---
## 📝 Summary

| Category | Tests | Key Verification |
|----------|-------|-------------------|
| Resume Checker | 1.1, 1.2 | Routes without JD, returns score/strengths/weaknesses |
| Resume Scorer | 2.1, 2.2 | Routes with JD, returns ATS analysis |
| Cover Letter | 3.1 | Routes with JD, returns 300-450 word letter |
| Career Coach | 4.1, 4.2 | Routes for advice queries, conversational |
| Multi-Tool | 5.1, 5.2 | Chains 3 tools, combined output with section headers |
| Multi without JD | 6.1 | Filters JD-dependent tools, only runs checker |
| Missing JD | 7.1, 7.2 | Returns \"need JD\" message |
| Memory | 8 | Follow-up references prior context |
| Ambiguous | 9.1, 9.2 | Falls back to career coach, no crash |